# Lab 05 — Task 2 (Easy English Version)

 Questions 1–12 



In [ ]:
# STEP 1: Upload and read the two CSV files
from google.colab import files
files.upload()

import pandas as pd
import heapq
from collections import deque

network = pd.read_csv("lab05_task2_network_times.csv")
devices = pd.read_csv("lab05_task2_device_names.csv")


names = dict(zip(devices["id"], devices["name"]))


graph = {}
for device in names:
    graph[device] = {}


for i in range(len(network)):
    a = network["device_a"].iloc[i]  
    b = network["device_b"].iloc[i]  
    time = int(network["minutes"].iloc[i])

    
    graph[a][b] = time
    graph[b][a] = time

print("Devices:", len(graph))
print("Connections:", len(network))

## A. Dijkstra, timeline, and infection counts (Questions 1–3)

In [ ]:
# STEP 2: One Dijkstra function, used again later

def dijkstra(start="N28", blocked=None):
    queue = [(0, start, None)]
    times = {}
    parent = {}

    while queue:
        time, device, previous = heapq.heappop(queue)
        if device in times or device == blocked:
            continue

        times[device] = time
        parent[device] = previous

        for neighbor, minutes in graph[device].items():
            if neighbor not in times and neighbor != blocked:
                heapq.heappush(queue, (time + minutes, neighbor, device))

    return times, parent


def path_to(device, parent):
    path = []
    while device is not None:
        path.append(names[device])
        device = parent[device]
    path.reverse()
    return " -> ".join(path)

# Q1 and Q2: All infection times and paths
times, parent = dijkstra()
print("ID | Device | Minutes | Path")
for device in sorted(times, key=times.get):
    print(device, "|", names[device], "|", times[device], "|", path_to(device, parent))

print("\nDatabase time:", times["N08"])
print("Database path:", path_to("N08", parent))
last = max(times, key=times.get)
print("Last infected:", names[last], times[last], "minutes")

# Q3: Devices infected within a given time
def infected_within(graph, start, minutes):
    infection_times, _ = dijkstra(start)
    found = []
    for device in infection_times:
        if infection_times[device] <= minutes:
            found.append(device)
    return found

for minute in [30, 60, 90]:
    found = infected_within(graph, "N28", minute)
    print(minute, "minutes:", len(found), "devices")
    if minute == 30:
        for device in found:
            print("  ", names[device])

## B. BFS vs Dijkstra (Questions 4–6)

In [ ]:
# STEP 3: BFS counts hops (connections), not minutes
queue = deque(["N28"])
hops = {"N28": 0}

while queue:
    device = queue.popleft()
    for neighbor in graph[device]:
        if neighbor not in hops:
            hops[neighbor] = hops[device] + 1
            queue.append(neighbor)

# Q4: Compare hops and infection times
for device in sorted(times, key=times.get):
    print(names[device], "| Hops:", hops[device], "| Minutes:", times[device])

# Q5: Find pairs where fewer hops take more minutes
pairs = []
for x in times:
    for y in times:
        if hops[x] < hops[y] and times[x] > times[y]:
            pairs.append((x, y))

print("\nTotal pairs:", len(pairs))
for x, y in pairs[:3]:
    print(names[x], "(", hops[x], "hops,", times[x], "min )")
    print(names[y], "(", hops[y], "hops,", times[y], "min )")

## C. Infection tree and containment (Questions 7–9)

In [ ]:
# STEP 4: Infection tree
# Q7: Print children below their parent (4 spaces per level)
def print_tree(device, level=0):
    print("    " * level + names[device])
    for child in parent:
        if parent[child] == device:
            print_tree(child, level + 1)

print_tree("N28")
print("Database infected by:", names[parent["N08"]])

# Q8: Count every child and grandchild
def count_descendants(device):
    total = 0
    for child in parent:
        if parent[child] == device:
            total += 1 + count_descendants(child)
    return total

counts = {}
for device in parent:
    counts[device] = count_descendants(device)

print("\nTop 5 spreaders:")
for device in sorted(counts, key=counts.get, reverse=True)[:5]:
    print(names[device], counts[device], "descendants")

# Q9: Remove each device before infection begins, one at a time
results = []
for blocked in graph:
    if blocked == "N28":
        continue
    remaining, _ = dijkstra(blocked=blocked)
    count = len(remaining)
    db_time = remaining.get("N08", "never")
    results.append((count, blocked, db_time))

print("\nRemoved | Infected | Database time")
for count, blocked, db_time in sorted(results):
    print(names[blocked], "|", count, "|", db_time)

print("\nDevices that limit infection to 7 or fewer:")
for count, blocked, db_time in sorted(results):
    if count <= 7:
        print(names[blocked], count)

## D. SOC response and Build-Server isolation (Questions 10–12)

In [ ]:
# STEP 5: SOC alert and Build-Server isolation
# Q10: At minute 40
infected40 = []
for device in times:
    if times[device] <= 40:
        infected40.append(device)

print("Infected at minute 40:", len(infected40))
for device in infected40:
    print(names[device], times[device], "minutes")
print("Still clean:", len(graph) - len(infected40))

# Q11: Remove Build-Server (N23) and run Dijkstra again

new_times, new_parent = dijkstra(blocked="N23")
new_db = new_times.get("N08")
print("\nOriginal database time:", times["N08"])
if new_db is None:
    print("Database never infected")
else:
    print("New database time:", new_db)
    print("Minutes gained:", new_db - times["N08"])
    print("New path:", path_to("N08", new_parent))

## Q12 — SOC recommendation 

1. Before the infection starts, removing WiFi-AP limits the spread to 1 device.
2. Removing Switch-Lab limits the spread to 3 devices.
3. Removing Core-Switch-2 limits the spread to 7 devices and protects Database-Server.
4. At minute 40, 12 devices are infected; the SOC should contain affected network segments quickly.
5. Database-Server is infected at minute 50 in the original simulation, 10 minutes after the alert.
6. Removing Build-Server alone delays infection by 4 minutes, so it does not fully protect the database.
